In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
torch.manual_seed(42)
from sklearn.metrics import f1_score

In [2]:
def read_data(filepath):
    sentences = [[]]
    for line in open(filepath):
        line = line.strip()
        if line == "":
            sentences.append([])
        else:
            try:
                idx, word, tag = line.split()
            except ValueError:
                idx, word = line.split()
                tag = "O"
            sentences[-1].append((word, tag))
    if sentences[-1] == []:
        sentences.pop()
    return sentences

In [3]:
train_sentences = read_data(r"/home/omghag/CSCI-544-Assignment/HW3/data/train")

In [4]:
print(f"Number of sentences: {len(train_sentences)}")
print(f"First sentence: {train_sentences[0]}")
print(f"Last sentence: {train_sentences[-1]}")

Number of sentences: 14987
First sentence: [('EU', 'B-ORG'), ('rejects', 'O'), ('German', 'B-MISC'), ('call', 'O'), ('to', 'O'), ('boycott', 'O'), ('British', 'B-MISC'), ('lamb', 'O'), ('.', 'O')]
Last sentence: [('-DOCSTART-', 'O')]


In [5]:
train_sentences = [s for s in train_sentences if s[0][0] != '-DOCSTART-']

In [6]:
print(f"Number of sentences: {len(train_sentences)}")
print(f"First sentence: {train_sentences[0]}")
print(f"Last sentence: {train_sentences[-1]}")

Number of sentences: 14041
First sentence: [('EU', 'B-ORG'), ('rejects', 'O'), ('German', 'B-MISC'), ('call', 'O'), ('to', 'O'), ('boycott', 'O'), ('British', 'B-MISC'), ('lamb', 'O'), ('.', 'O')]
Last sentence: [('Swansea', 'B-ORG'), ('1', 'O'), ('Lincoln', 'B-ORG'), ('2', 'O')]


In [7]:
test_sentences = read_data(r"/home/omghag/CSCI-544-Assignment/HW3/data/test")
print(f"Number of sentences: {len(test_sentences)}")
print(f"First sentence: {test_sentences[0]}")
print(f"Last sentence: {test_sentences[-1]}")

Number of sentences: 3684
First sentence: [('SOCCER', 'O'), ('-', 'O'), ('JAPAN', 'O'), ('GET', 'O'), ('LUCKY', 'O'), ('WIN', 'O'), (',', 'O'), ('CHINA', 'O'), ('IN', 'O'), ('SURPRISE', 'O'), ('DEFEAT', 'O'), ('.', 'O')]
Last sentence: [('-DOCSTART-', 'O')]


In [8]:
test_sentences = [s for s in test_sentences if s[0][0] != '-DOCSTART-']

In [9]:
print(f"Number of sentences: {len(test_sentences)}")
print(f"First sentence: {test_sentences[0]}")
print(f"Last sentence: {test_sentences[-1]}")

Number of sentences: 3453
First sentence: [('SOCCER', 'O'), ('-', 'O'), ('JAPAN', 'O'), ('GET', 'O'), ('LUCKY', 'O'), ('WIN', 'O'), (',', 'O'), ('CHINA', 'O'), ('IN', 'O'), ('SURPRISE', 'O'), ('DEFEAT', 'O'), ('.', 'O')]
Last sentence: [('The', 'O'), ('lanky', 'O'), ('former', 'O'), ('Leeds', 'O'), ('United', 'O'), ('defender', 'O'), ('did', 'O'), ('not', 'O'), ('make', 'O'), ('his', 'O'), ('England', 'O'), ('debut', 'O'), ('until', 'O'), ('the', 'O'), ('age', 'O'), ('of', 'O'), ('30', 'O'), ('but', 'O'), ('eventually', 'O'), ('won', 'O'), ('35', 'O'), ('caps', 'O'), ('and', 'O'), ('was', 'O'), ('a', 'O'), ('key', 'O'), ('member', 'O'), ('of', 'O'), ('the', 'O'), ('1966', 'O'), ('World', 'O'), ('Cup', 'O'), ('winning', 'O'), ('team', 'O'), ('with', 'O'), ('his', 'O'), ('younger', 'O'), ('brother', 'O'), (',', 'O'), ('Bobby', 'O'), ('.', 'O')]


In [10]:
dev_sentences = read_data(r"/home/omghag/CSCI-544-Assignment/HW3/data/dev")
print(f"Number of sentences: {len(dev_sentences)}")
print(f"First sentence: {dev_sentences[0]}")
print(f"Last sentence: {dev_sentences[-1]}")

Number of sentences: 3466
First sentence: [('CRICKET', 'O'), ('-', 'O'), ('LEICESTERSHIRE', 'B-ORG'), ('TAKE', 'O'), ('OVER', 'O'), ('AT', 'O'), ('TOP', 'O'), ('AFTER', 'O'), ('INNINGS', 'O'), ('VICTORY', 'O'), ('.', 'O')]
Last sentence: [('-DOCSTART-', 'O')]


In [11]:
dev_sentences = [s for s in dev_sentences if s[0][0] != '-DOCSTART-']
print(f"Number of sentences: {len(dev_sentences)}")
print(f"First sentence: {dev_sentences[0]}")
print(f"Last sentence: {dev_sentences[-1]}")

Number of sentences: 3250
First sentence: [('CRICKET', 'O'), ('-', 'O'), ('LEICESTERSHIRE', 'B-ORG'), ('TAKE', 'O'), ('OVER', 'O'), ('AT', 'O'), ('TOP', 'O'), ('AFTER', 'O'), ('INNINGS', 'O'), ('VICTORY', 'O'), ('.', 'O')]
Last sentence: [('--', 'O'), ('Dhaka', 'B-ORG'), ('Newsroom', 'I-ORG'), ('880-2-506363', 'O')]


In [12]:
def build_vocab(sentences):
    word2idx = {"<PAD>": 0, "<UNK>": 1}
    tag2idx = {"<PAD>": 0}
    
    for sentence in sentences:
        for word, tag in sentence:
            if word not in word2idx:
                word2idx[word] = len(word2idx)
            if tag not in tag2idx:
                tag2idx[tag] = len(tag2idx)
    
    return word2idx, tag2idx

In [13]:
word2idx, tag2idx = build_vocab(train_sentences)
print(f"Number of unique words: {len(word2idx)}")
print(f"Number of unique tags: {len(tag2idx)}")
tag2idx

Number of unique words: 23625
Number of unique tags: 10


{'<PAD>': 0,
 'B-ORG': 1,
 'O': 2,
 'B-MISC': 3,
 'B-PER': 4,
 'I-PER': 5,
 'B-LOC': 6,
 'I-ORG': 7,
 'I-MISC': 8,
 'I-LOC': 9}

In [14]:
idx2tag = {v: k for k, v in tag2idx.items()}

In [15]:
def encode_data(sentences, word2idx, tag2idx):
    word_ids = [[word2idx.get(word, word2idx["<UNK>"]) for word, tag in sentence] for sentence in sentences]
    tag_ids = [[tag2idx[tag] for word, tag in sentence] for sentence in sentences]
    return word_ids, tag_ids

In [16]:
train_words, train_tags = encode_data(train_sentences, word2idx, tag2idx)
dev_words, dev_tags = encode_data(dev_sentences, word2idx, tag2idx)
test_words, test_tags = encode_data(test_sentences, word2idx, tag2idx)

In [17]:
class NERDataset(Dataset):
    def __init__(self, word_ids, tag_ids):
        self.word_ids = word_ids
        self.tag_ids = tag_ids

    def __len__(self):
        return len(self.word_ids)

    def __getitem__(self, idx):
        return torch.tensor(self.word_ids[idx]), torch.tensor(self.tag_ids[idx])

In [18]:
def collate_fn(batch):
    word_ids, tag_ids = zip(*batch)
    word_ids_padded = pad_sequence(word_ids, batch_first=True, padding_value=0)
    tag_ids_padded = pad_sequence(tag_ids, batch_first=True, padding_value=0)
    return word_ids_padded, tag_ids_padded

In [19]:
train_dataset = NERDataset(train_words, train_tags)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

In [20]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(NERDataset(dev_words, dev_tags), batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(NERDataset(test_words, test_tags), batch_size=16, shuffle=False, collate_fn=collate_fn)

In [21]:
import torch.nn as nn

class BLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, 
                 dropout, linear_dim, num_tags):
        super(BLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.blstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, 
                            dropout=dropout, bidirectional=True, batch_first=True)
        self.linear = nn.Linear(hidden_dim * 2, linear_dim)
        self.elu = nn.ELU()
        self.classifier = nn.Linear(linear_dim, num_tags)
    
    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.blstm(x)
        x = self.linear(x)
        x = self.elu(x)
        x = self.classifier(x)
        return x


In [22]:
model = BLSTM(
    vocab_size=len(word2idx),
    embedding_dim=100,
    hidden_dim=256,
    num_layers=1,
    dropout=0.33,
    linear_dim=128,
    num_tags=len(tag2idx)
)

/home/omghag/CSCI-544-Assignment/.venv/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.33 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [23]:
words, tags = next(iter(train_loader))
print(f"Input shape: {words.shape}")
output = model(words)
print(f"Output shape: {output.shape}")

Input shape: torch.Size([16, 39])
Output shape: torch.Size([16, 39, 10])


In [24]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model= model.to(device)

In [25]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for words, tags in loader:
        words = words.to(device)
        tags = tags.to(device)
        # 1. zero gradients
        optimizer.zero_grad()
        # 2. forward pass
        output = model(words)
        # 3. reshape output and tags
        output = output.view(-1, output.size(-1))
        tags = tags.view(-1)
        # 4. compute loss
        loss = criterion(output, tags)
        # 5. backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        # 6. update weights
        optimizer.step()
        # 7. accumulate loss
        total_loss += loss.item()
    return total_loss / len(loader)

In [26]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds = []
    all_tags = []
    
    with torch.no_grad():
        for words, tags in loader:
            words = words.to(device)
            tags = tags.to(device)
            # forward pass + loss
            output = model(words)
            loss = criterion(output.view(-1, output.size(-1)), tags.view(-1))
            total_loss += loss.item()

            # collect predictions using argmax
            preds = torch.argmax(output, dim=-1)
            
            for pred_seq, tag_seq in zip(preds, tags):
                for p, t in zip(pred_seq, tag_seq):
                    if t != 0:  # ignore padding
                        all_preds.append(p.item())
                        all_tags.append(t.item())


    return total_loss / len(loader), all_preds, all_tags

In [27]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.3)
criterion = nn.CrossEntropyLoss(ignore_index=0)

In [28]:
num_epochs = 10

for epoch in range(num_epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    val_loss, preds, true_tags = evaluate(model, dev_loader, criterion)
    scheduler.step(val_loss)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

torch.save(model.state_dict(), 'blstm1.pt')

Epoch 1/10 | Train Loss: 0.5374 | Val Loss: 0.3858
Epoch 2/10 | Train Loss: 0.2611 | Val Loss: 0.3072
Epoch 3/10 | Train Loss: 0.1498 | Val Loss: 0.2264
Epoch 4/10 | Train Loss: 0.0882 | Val Loss: 0.2279
Epoch 5/10 | Train Loss: 0.0490 | Val Loss: 0.2473
Epoch 6/10 | Train Loss: 0.0259 | Val Loss: 0.2801
Epoch 7/10 | Train Loss: 0.0079 | Val Loss: 0.2788
Epoch 8/10 | Train Loss: 0.0031 | Val Loss: 0.2882
Epoch 9/10 | Train Loss: 0.0022 | Val Loss: 0.2959
Epoch 10/10 | Train Loss: 0.0015 | Val Loss: 0.3036


In [29]:
loss, preds, true_tags = evaluate(model, dev_loader, criterion)
print(f"Loss: {loss:.4f}")
pred_tags = [idx2tag[p] for p in preds]
true_tag_strings = [idx2tag[t] for t in true_tags]

Loss: 0.3036


In [30]:
# how many O tags vs B-PER, I-LOC etc in your dataset?
from collections import Counter
all_train_tags = [tag for sent in train_sentences for _, tag in sent]
print(Counter(all_train_tags))

Counter({'O': 169578, 'B-LOC': 7140, 'B-PER': 6600, 'B-ORG': 6321, 'I-PER': 4528, 'I-ORG': 3704, 'B-MISC': 3438, 'I-LOC': 1157, 'I-MISC': 1155})


In [31]:
dev_sentences_raw = read_data("/home/omghag/CSCI-544-Assignment/HW3/data/dev")  # no DOCSTART filter
test_sentences_raw = read_data("/home/omghag/CSCI-544-Assignment/HW3/data/test")  # no DOCSTART filter

In [32]:
def write_predictions(sentences, preds, idx2tag, output_file, include_docstart=True):
    with open(output_file, "w") as f:
        flat_idx = 0
        for sentence in sentences:
            if include_docstart and sentence[0][0] == '-DOCSTART-':
                f.write("1 -DOCSTART- O\n\n")
                continue
            for i, (word, _) in enumerate(sentence):
                pred_tag = idx2tag[preds[flat_idx]]
                f.write(f"{i+1} {word} {pred_tag}\n")
                flat_idx += 1
            f.write("\n")

In [33]:
write_predictions(dev_sentences_raw, preds ,idx2tag, "dev1.out")

In [34]:
loss, preds, true_tags = evaluate(model, test_loader, criterion)
print(f"Loss: {loss:.4f}")
pred_tags = [idx2tag[p] for p in preds]
true_tag_strings = [idx2tag[t] for t in true_tags]
write_predictions(test_sentences_raw, preds ,idx2tag, "test1.out")

Loss: 1.8672


In [35]:
model.load_state_dict(torch.load('/home/omghag/CSCI-544-Assignment/HW3/blstm1.pt'))

<All keys matched successfully>

In [36]:
def get_cap_feature(word):
    # returns 0, 1, 2, or 3
    # 0 = all lowercase
    if word.islower():
        return 0
    # 1 = all caps
    elif word.isupper():
        return 1
    # 2 = title case
    elif word.istitle():
        return 2
    # 3 = other/mixed
    else:
        return 3

In [37]:
import gzip
import numpy as np

def load_glove(glove_path, word2idx, embedding_dim=100):
    # step 1: initialize embedding matrix with zeros or random
    embedding_matrix = np.zeros((len(word2idx), embedding_dim))
    
    # step 2: load glove vectors into a dict
    glove_dict = {}
    with gzip.open(glove_path, 'rt', encoding='utf-8') as f:
        for line in f:
            # each line is: "word 0.1 0.2 ... 0.n"
            # parse it here
            parts = line.strip().split()
            word = parts[0]
            vector = np.array([float(x) for x in parts[1:]])
            glove_dict[word] = vector
    
    # step 3: compute average vector for UNK
    avg_vector = np.mean(list(glove_dict.values()), axis=0)
    
    # step 4: fill embedding matrix
    for word, idx in word2idx.items():
        # use glove vector if available, else avg_vector
        embedding_matrix[idx] = glove_dict.get(word.lower(), avg_vector)
    
    return embedding_matrix

In [38]:
glove_path = r"/home/omghag/CSCI-544-Assignment/HW3/glove.6B.100d.gz"
embedding_matrix = load_glove(glove_path, word2idx)
print(embedding_matrix.shape)  # should be (23625, 100)

(23625, 100)


In [39]:
import gzip

with gzip.open(glove_path, 'rt', encoding='utf-8', errors='ignore') as f:
    print(f.readline())

the -0.038194 -0.24487 0.72812 -0.39961 0.083172 0.043953 -0.39141 0.3344 -0.57545 0.087459 0.28787 -0.06731 0.30906 -0.26384 -0.13231 -0.20757 0.33395 -0.33848 -0.31743 -0.48336 0.1464 -0.37304 0.34577 0.052041 0.44946 -0.46971 0.02628 -0.54155 -0.15518 -0.14107 -0.039722 0.28277 0.14393 0.23464 -0.31021 0.086173 0.20397 0.52624 0.17164 -0.082378 -0.71787 -0.41531 0.20335 -0.12763 0.41367 0.55187 0.57908 -0.33477 -0.36559 -0.54857 -0.062892 0.26584 0.30205 0.99775 -0.80481 -3.0243 0.01254 -0.36942 2.2167 0.72201 -0.24978 0.92136 0.034514 0.46745 1.1079 -0.19358 -0.074575 0.23353 -0.052062 -0.22044 0.057162 -0.15806 -0.30798 -0.41625 0.37972 0.15006 -0.53212 -0.2055 -1.2526 0.071624 0.70565 0.49744 -0.42063 0.26148 -1.538 -0.30223 -0.073438 -0.28312 0.37104 -0.25217 0.016215 -0.017099 -0.38984 0.87424 -0.72569 -0.51058 -0.52028 -0.1459 0.8278 0.27062



In [40]:
class BLSTM2(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers,
                 dropout, linear_dim, num_tags, embedding_matrix, cap_size, cap_embedding_dim):
        super(BLSTM2, self).__init__()
        
        # GloVe embedding
        self.embedding = nn.Embedding.from_pretrained(torch.FloatTensor(embedding_matrix), freeze=False, padding_idx=0)
        
        # capitalization embedding
        self.cap_embedding = nn.Embedding(cap_size, cap_embedding_dim)
        
        # BLSTM input dim is now embedding_dim + cap_embedding_dim
        self.blstm = nn.LSTM(embedding_dim + cap_embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout, bidirectional=True)
        
        self.linear = nn.Linear(hidden_dim * 2, linear_dim)
        self.elu = nn.ELU()
        self.classifier = nn.Linear(linear_dim, num_tags)
    
    def forward(self, x, cap_ids):
        x = self.embedding(x)
        x = torch.cat((x, self.cap_embedding(cap_ids)), dim=-1)  # concatenate word and cap embeddings
        x, _ = self.blstm(x)
        x = self.linear(x)
        x = self.elu(x)
        x = self.classifier(x)
        return x

In [41]:
def encode_cap_features(sentences):
    cap_ids = [[get_cap_feature(word) for word, tag in sentence] 
               for sentence in sentences]
    return cap_ids

In [42]:
class NERDataset2(Dataset):
    def __init__(self, word_ids, tag_ids, cap_ids):
        self.word_ids = word_ids
        self.tag_ids = tag_ids
        self.cap_ids = cap_ids

    def __len__(self):
        return len(self.word_ids)

    def __getitem__(self, idx):
        return torch.tensor(self.word_ids[idx]), torch.tensor(self.tag_ids[idx]), torch.tensor(self.cap_ids[idx])

In [43]:
def collate_fn2(batch):
    word_ids, tag_ids, cap_ids = zip(*batch)
    word_ids_padded = pad_sequence(word_ids, batch_first=True, padding_value=0)
    tag_ids_padded = pad_sequence(tag_ids, batch_first=True, padding_value=0)
    cap_ids_padded = pad_sequence(cap_ids, batch_first=True, padding_value=0)
    return word_ids_padded, tag_ids_padded, cap_ids_padded

In [44]:
def train2(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for words, tags, cap in loader:
        words = words.to(device)
        tags = tags.to(device)
        cap = cap.to(device)
        # 1. zero gradients
        optimizer.zero_grad()
        # 2. forward pass
        output = model(words, cap)
        # 3. reshape output and tags
        output = output.view(-1, output.size(-1))
        tags = tags.view(-1)
        # 4. compute loss
        loss = criterion(output, tags)
        # 5. backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        # 6. update weights
        optimizer.step()
        # 7. accumulate loss
        total_loss += loss.item()
    return total_loss / len(loader)

In [45]:
def evaluate2(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds = []
    all_tags = []
    
    with torch.no_grad():
        for words, tags ,cap in loader:
            words = words.to(device)
            tags = tags.to(device)
            cap = cap.to(device)
            # forward pass + loss
            output = model(words, cap)
            loss = criterion(output.view(-1, output.size(-1)), tags.view(-1))
            total_loss += loss.item()

            # collect predictions using argmax
            preds = torch.argmax(output, dim=-1)
            
            for pred_seq, tag_seq in zip(preds, tags):
                for p, t in zip(pred_seq, tag_seq):
                    if t != 0:  # ignore padding
                        all_preds.append(p.item())
                        all_tags.append(t.item())


    return total_loss / len(loader), all_preds, all_tags

In [46]:
# encode cap features
train_caps = encode_cap_features(train_sentences)
dev_caps = encode_cap_features(dev_sentences)
test_caps = encode_cap_features(test_sentences)

# create datasets
train_dataset2 = NERDataset2(train_words, train_tags, train_caps)
dev_dataset2 = NERDataset2(dev_words, dev_tags, dev_caps)
test_dataset2 = NERDataset2(test_words, test_tags, test_caps)

# dataloaders
train_loader2 = DataLoader(train_dataset2, batch_size=16, shuffle=True, collate_fn=collate_fn2)
dev_loader2 = DataLoader(dev_dataset2, batch_size=16, shuffle=False, collate_fn=collate_fn2)
test_loader2 = DataLoader(test_dataset2, batch_size=16, shuffle=False, collate_fn=collate_fn2)

# model
model2 = BLSTM2(
    vocab_size=len(word2idx),
    embedding_dim=100,
    hidden_dim=256,
    num_layers=1,
    dropout=0.33,
    linear_dim=128,
    num_tags=len(tag2idx),
    embedding_matrix=embedding_matrix,
    cap_size=4,
    cap_embedding_dim=16
).to(device)

optimizer2 = torch.optim.SGD(model2.parameters(), lr=0.1, momentum=0.9)
scheduler2 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer2, patience=2, factor=0.3)
criterion2 = nn.CrossEntropyLoss(ignore_index=0)

/home/omghag/CSCI-544-Assignment/.venv/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.33 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [47]:
num_epochs = 10

for epoch in range(num_epochs):
    train_loss = train2(model2, train_loader2, optimizer2, criterion2)
    val_loss, preds, true_tags = evaluate2(model2, dev_loader2, criterion2)
    scheduler2.step(val_loss)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

torch.save(model2.state_dict(), 'blstm2.pt')

Epoch 1/10 | Train Loss: 0.1805 | Val Loss: 0.1270
Epoch 2/10 | Train Loss: 0.0809 | Val Loss: 0.1059
Epoch 3/10 | Train Loss: 0.0613 | Val Loss: 0.0838
Epoch 4/10 | Train Loss: 0.0494 | Val Loss: 0.0806
Epoch 5/10 | Train Loss: 0.0407 | Val Loss: 0.0776
Epoch 6/10 | Train Loss: 0.0331 | Val Loss: 0.0758
Epoch 7/10 | Train Loss: 0.0265 | Val Loss: 0.0739
Epoch 8/10 | Train Loss: 0.0227 | Val Loss: 0.0700
Epoch 9/10 | Train Loss: 0.0181 | Val Loss: 0.0755
Epoch 10/10 | Train Loss: 0.0141 | Val Loss: 0.0901


In [48]:
loss, preds, true_tags = evaluate2(model2, dev_loader2, criterion2)
print(f"Loss: {loss:.4f}")
pred_tags = [idx2tag[p] for p in preds]
true_tag_strings = [idx2tag[t] for t in true_tags]

Loss: 0.0901


In [49]:
write_predictions(dev_sentences_raw, preds ,idx2tag, "dev2.out")

In [50]:
loss, preds, true_tags = evaluate2(model2, test_loader2, criterion2)
print(f"Loss: {loss:.4f}")
pred_tags = [idx2tag[p] for p in preds]
true_tag_strings = [idx2tag[t] for t in true_tags]

Loss: 2.0476


In [51]:
test_sentences_raw = read_data("/home/omghag/CSCI-544-Assignment/HW3/data/test")  # no DOCSTART filter
write_predictions(test_sentences_raw, preds ,idx2tag, "test2.out")

In [52]:
def build_char_vocab(sentences):
    char2idx = {"<PAD>": 0, "<UNK>": 1}
    for sentence in sentences:
        for word, tag in sentence:
            for char in word:
                if char not in char2idx:
                    char2idx[char] = len(char2idx)
    return char2idx

In [53]:
def word_to_char_ids(word, char2idx, max_word_len=20):
    char_ids = [char2idx.get(c, char2idx["<UNK>"]) for c in word[:max_word_len]]
    # pad with zeros if shorter than max_word_len
    char_ids += [char2idx["<PAD>"]] * (max_word_len - len(char_ids))
    return char_ids

In [54]:
def encode_char_data(sentences, char2idx, max_word_len=20):
    # returns list of lists of char_id lists
    # each sentence → list of words → list of char ids
    encoded = []
    for sentence in sentences:
        word_encoded = []
        for word, tag in sentence:
            char_ids = [char2idx.get(c, char2idx['<UNK>']) for c in word[:max_word_len]]
            char_ids += [char2idx['<PAD>']] * (max_word_len - len(char_ids))
            word_encoded.append(char_ids)
        encoded.append(word_encoded)
    return encoded

In [55]:
char2idx = build_char_vocab(train_sentences)
print(f"Char vocab size: {len(char2idx)}")

train_chars = encode_char_data(train_sentences, char2idx)
dev_chars = encode_char_data(dev_sentences, char2idx)
test_chars = encode_char_data(test_sentences, char2idx)

Char vocab size: 86


In [56]:
class NERDatasetCNN(Dataset):
    def __init__(self, word_ids, tag_ids, cap_ids, char_ids):
        self.word_ids = word_ids
        self.tag_ids = tag_ids
        self.cap_ids = cap_ids
        self.char_ids = char_ids

    def __len__(self):
        return len(self.word_ids)

    def __getitem__(self, idx):
        return (torch.tensor(self.word_ids[idx]), 
                torch.tensor(self.tag_ids[idx]), 
                torch.tensor(self.cap_ids[idx]), 
                torch.tensor(self.char_ids[idx]))

In [57]:
def collate_fn_cnn(batch):
    word_ids, tag_ids, cap_ids, char_ids = zip(*batch)
    
    word_ids_padded = pad_sequence(word_ids, batch_first=True, padding_value=0)
    tag_ids_padded = pad_sequence(tag_ids, batch_first=True, padding_value=0)
    cap_ids_padded = pad_sequence(cap_ids, batch_first=True, padding_value=0)
    
    # char_ids: each element is (seq_len, max_word_len)
    # pad_sequence will pad along seq_len dimension automatically!
    char_ids_padded = pad_sequence(char_ids, batch_first=True, padding_value=0)
    # result shape: (batch_size, max_seq_len, max_word_len)
    
    return word_ids_padded, tag_ids_padded, cap_ids_padded, char_ids_padded

In [58]:
train_dataset_cnn = NERDatasetCNN(train_words, train_tags, train_caps, train_chars)
dev_dataset_cnn = NERDatasetCNN(dev_words, dev_tags, dev_caps, dev_chars)
test_dataset_cnn = NERDatasetCNN(test_words, test_tags, test_caps, test_chars)

In [59]:
train_loader_cnn = DataLoader(train_dataset_cnn, batch_size=16, shuffle=True, collate_fn=collate_fn_cnn)
dev_loader_cnn = DataLoader(dev_dataset_cnn, batch_size=16, shuffle=False, collate_fn=collate_fn_cnn)
test_loader_cnn = DataLoader(test_dataset_cnn, batch_size=16, shuffle=False, collate_fn=collate_fn_cnn)

In [60]:
words, tags, caps, chars = next(iter(train_loader_cnn))
print(f"words: {words.shape}")
print(f"chars: {chars.shape}")

words: torch.Size([16, 42])
chars: torch.Size([16, 42, 20])


In [61]:
class CNNBLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers,
                 dropout, linear_dim, num_tags, embedding_matrix, cap_size, cap_embedding_dim,
                 char_vocab_size, char_embedding_dim, char_cnn_out_channels, char_cnn_kernel_size):
        super(CNNBLSTM, self).__init__()
        
        self.embedding = nn.Embedding.from_pretrained(torch.FloatTensor(embedding_matrix), freeze=False, padding_idx=0)
        self.cap_embedding = nn.Embedding(cap_size, cap_embedding_dim)
        self.char_embedding = nn.Embedding(char_vocab_size, char_embedding_dim, padding_idx=0)
        self.char_cnn = nn.Conv1d(in_channels=char_embedding_dim, out_channels=char_cnn_out_channels, kernel_size=char_cnn_kernel_size, padding=char_cnn_kernel_size//2)
        self.blstm = nn.LSTM(embedding_dim + cap_embedding_dim + char_cnn_out_channels, hidden_dim, num_layers, batch_first=True, dropout=dropout, bidirectional=True)
        self.linear = nn.Linear(hidden_dim * 2, linear_dim)
        self.elu = nn.ELU()
        self.classifier = nn.Linear(linear_dim, num_tags)
        
    def forward(self, x, cap_ids, char_ids):
        batch_size, seq_len, max_word_len = char_ids.size()
        char_ids = char_ids.view(-1, max_word_len)  # (batch_size*seq_len, max_word_len)
        char_embeds = self.char_embedding(char_ids)  # (batch*seq_len, max_word_len, char_dim)
        char_embeds = char_embeds.permute(0, 2, 1)   # (batch*seq_len, char_dim, max_word_len)
        char_cnn_out = self.char_cnn(char_embeds)     # (batch*seq_len, num_filters, max_word_len)
        char_cnn_out = torch.max(char_cnn_out, dim=2)[0]  # (batch*seq_len, num_filters)
        char_cnn_out = char_cnn_out.view(batch_size, seq_len, -1)  # (batch_size, seq_len, char_cnn_out_channels)

        x = self.embedding(x)
        cap_embeds = self.cap_embedding(cap_ids)
        x = torch.cat((x, cap_embeds, char_cnn_out), dim=-1)  # concatenate word, cap and char features
        x, _ = self.blstm(x)
        x = self.linear(x)
        x = self.elu(x)
        x = self.classifier(x)
        return x

In [71]:
model_cnn = CNNBLSTM(
    vocab_size=len(word2idx),
    embedding_dim=100,
    hidden_dim=256,
    num_layers=1,
    dropout=0.33,
    linear_dim=128,
    num_tags=len(tag2idx),
    embedding_matrix=embedding_matrix,
    cap_size=4,
    cap_embedding_dim=16,
    char_vocab_size=len(char2idx),
    char_embedding_dim=30,  
    char_cnn_out_channels=100,
    char_cnn_kernel_size=3
).to(device)

/home/omghag/CSCI-544-Assignment/.venv/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.33 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [72]:
words, tags, caps, chars = next(iter(train_loader_cnn))
words, tags, caps, chars = words.to(device), tags.to(device), caps.to(device), chars.to(device)

output = model_cnn(words, caps, chars)
print(output.shape)  # what do you expect?

torch.Size([16, 41, 10])


In [73]:
def train_cnn(model_cnn, loader, optimizer, criterion):
    model_cnn.train()
    total_loss = 0
    for words, tags, caps, chars in loader:
        words = words.to(device)
        tags = tags.to(device)
        caps = caps.to(device)
        chars = chars.to(device)
        
        optimizer.zero_grad()
        output = model_cnn(words, caps, chars)
        output = output.view(-1, output.size(-1))
        tags = tags.view(-1)
        loss = criterion(output, tags)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_cnn.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [74]:
def evaluate_cnn(model_cnn, loader, criterion):
    model_cnn.eval()
    total_loss = 0
    all_preds = []
    all_tags = []
    
    with torch.no_grad():
        for words, tags, caps, chars in loader:
            words = words.to(device)
            tags = tags.to(device)
            caps = caps.to(device)
            chars = chars.to(device)
            
            output = model_cnn(words, caps, chars)
            loss = criterion(output.view(-1, output.size(-1)), tags.view(-1))
            total_loss += loss.item()

            preds = torch.argmax(output, dim=-1)
            
            for pred_seq, tag_seq in zip(preds, tags):
                for p, t in zip(pred_seq, tag_seq):
                    if t != 0:
                        all_preds.append(p.item())
                        all_tags.append(t.item())

    return total_loss / len(loader), all_preds, all_tags

In [75]:
optimizer_cnn = torch.optim.SGD(model_cnn.parameters(), lr=0.1, momentum=0.9)
scheduler_cnn = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_cnn, patience=2, factor=0.3)
criterion_cnn = nn.CrossEntropyLoss(ignore_index=0)

best_val_loss = float('inf')
patience_counter = 0
patience = 5
num_epochs = 30

for epoch in range(num_epochs):
    train_loss = train_cnn(model_cnn, train_loader_cnn, optimizer_cnn, criterion_cnn)
    val_loss, preds, true_tags = evaluate_cnn(model_cnn, dev_loader_cnn, criterion_cnn)
    scheduler_cnn.step(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model_cnn.state_dict(), 'blstm_cnn.pt')
        patience_counter = 0
        print(f"  ✓ Best model saved!")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print("Early stopping!")
            break

model_cnn.load_state_dict(torch.load('blstm_cnn.pt'))

Epoch 1/30 | Train Loss: 0.1762 | Val Loss: 0.1354
  ✓ Best model saved!
Epoch 2/30 | Train Loss: 0.0716 | Val Loss: 0.0933
  ✓ Best model saved!
Epoch 3/30 | Train Loss: 0.0526 | Val Loss: 0.0790
  ✓ Best model saved!
Epoch 4/30 | Train Loss: 0.0402 | Val Loss: 0.0806
  No improvement (1/5)
Epoch 5/30 | Train Loss: 0.0317 | Val Loss: 0.0761
  ✓ Best model saved!
Epoch 6/30 | Train Loss: 0.0254 | Val Loss: 0.0743
  ✓ Best model saved!
Epoch 7/30 | Train Loss: 0.0188 | Val Loss: 0.0809
  No improvement (1/5)
Epoch 8/30 | Train Loss: 0.0151 | Val Loss: 0.0829
  No improvement (2/5)
Epoch 9/30 | Train Loss: 0.0112 | Val Loss: 0.0861
  No improvement (3/5)
Epoch 10/30 | Train Loss: 0.0052 | Val Loss: 0.0833
  No improvement (4/5)
Epoch 11/30 | Train Loss: 0.0031 | Val Loss: 0.0864
  No improvement (5/5)
Early stopping!


<All keys matched successfully>

In [78]:
loss, preds, true_tags = evaluate_cnn(model_cnn, dev_loader_cnn, criterion_cnn)
print(f"Loss: {loss:.4f}")
pred_tags = [idx2tag[p] for p in preds]
true_tag_strings = [idx2tag[t] for t in true_tags]
write_predictions(dev_sentences_raw, preds ,idx2tag, "dev_cnn.out")

Loss: 0.0743


In [77]:
loss, preds, true_tags = evaluate_cnn(model_cnn, test_loader_cnn, criterion_cnn)
print(f"Loss: {loss:.4f}")
pred_tags = [idx2tag[p] for p in preds]
true_tag_strings = [idx2tag[t] for t in true_tags]
write_predictions(test_sentences_raw, preds ,idx2tag, "test_cnn.out")

Loss: 2.0490
